## Prequest: Pipeline Used Before Running Secondary Analsysis
The input data was generated using the nf-lly-scrnaseq pipeline with the following command:

```bash
nextflow run ~/scripts/nf-lly-snrnaseq/main.nf \
  -profile pcluster \
  --sample_sheet test_nf-lly-snRNA.csv \
  --fasta ~/references/10xh38/refdata-gex-GRCh38-2024-A/fasta/genome.fa \
  --gtf ~/references/10xh38/refdata-gex-GRCh38-2024-A/genes/genes.gtf \
  --protocol auto \
  --aligner cellranger \
  --outdir output/
```

### Input Data
This analysis uses processed H5AD files from the pipeline's `mtx_conversions` step:
- **Filtered matrix:** `*_emptydrops_filter_matrix.h5ad` (filtered cells)
- **Raw matrix:** `*_raw_matrix.h5ad` (unfiltered matrix)

#### Example Input Paths
For sample `SRR14575500`:
```
output/cellranger/mtx_conversions/SRR14575500/SRR14575500_emptydrops_filter_matrix.h5ad
output/cellranger/mtx_conversions/SRR14575500/SRR14575500_raw_matrix.h5ad
```

In [ ]:
from pathlib import Path
import sys

# Run Jupyter with this project folder as its working directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts" / "project_paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts" / "project_paths.py").is_file():
    raise RuntimeError("Set the notebook working directory to the project folder.")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from project_paths import atlas_path, data_path


## Prepare h5ad by adding the metadata before running next step

In [ ]:
import scanpy as sc
from pathlib import Path

# Directory containing one folder per sample (e.g., SRR14575500/)
BASE_DIR = Path(str(data_path('cellranger/mtx_conversions')))

# Output directory for metadata-added h5ad files (kept separate from originals)
OUT_DIR = Path(str(data_path('meta_added_h5ad')))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Fill in metadata for each sample folder.
# Keys must match folder names under BASE_DIR.
sample_metadata = {
    "SRR14575500": {"treatment": "ctr", "batch": "batch1"},
    "SRR14575504": {"treatment": "pre-treatment", "batch": "batch1"},
    "SRR14575508": {"treatment": "post-treatment", "batch": "batch1"},
}

required_fields = {"treatment", "batch"}
missing_samples = [s for s in sample_metadata if not (BASE_DIR / s).is_dir()]
if missing_samples:
    raise FileNotFoundError(f"These sample folders are missing under {BASE_DIR}: {missing_samples}")


def update_and_save_h5ad(input_path: Path, output_path: Path, meta: dict):
    adata = sc.read_h5ad(input_path)
    for field in required_fields:
        adata.obs[field] = meta[field]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(output_path)


updated = []
skipped = []

for sample_dir in sorted([p for p in BASE_DIR.iterdir() if p.is_dir()]):
    sample = sample_dir.name

    # Ignore directories you have not explicitly annotated.
    if sample not in sample_metadata:
        skipped.append(sample)
        continue

    meta = sample_metadata[sample]
    if not required_fields.issubset(meta.keys()):
        missing_fields = required_fields - set(meta.keys())
        raise ValueError(f"Metadata for {sample} is missing fields: {missing_fields}")

    filtered_files = sorted(sample_dir.glob("*emptydrops_filter_matrix.h5ad"))
    raw_files = sorted(sample_dir.glob("*raw_matrix.h5ad"))

    if len(filtered_files) != 1:
        raise FileNotFoundError(
            f"Expected exactly 1 filtered h5ad in {sample_dir}, found {len(filtered_files)}: {filtered_files}"
        )
    if len(raw_files) != 1:
        raise FileNotFoundError(
            f"Expected exactly 1 unfiltered h5ad in {sample_dir}, found {len(raw_files)}: {raw_files}"
        )

    out_sample_dir = OUT_DIR / sample
    file_pairs = [
        (filtered_files[0], out_sample_dir / filtered_files[0].name, "filtered"),
        (raw_files[0], out_sample_dir / raw_files[0].name, "unfiltered"),
    ]

    for in_h5ad, out_h5ad, matrix_type in file_pairs:
        update_and_save_h5ad(in_h5ad, out_h5ad, meta)
        updated.append((sample, matrix_type, str(in_h5ad), str(out_h5ad)))

print(f"Wrote {len(updated)} metadata-added files across {len(sample_metadata)} samples.")
print(f"Output root: {OUT_DIR}")
if skipped:
    print("Skipped folders not in sample_metadata:", skipped)

for sample, matrix_type, in_h5ad, out_h5ad in updated:
    print(f"[{sample}] {matrix_type}: {in_h5ad} -> {out_h5ad}")

### Step 1.1 QC Raw — Per-Sample Metrics

In [ ]:
import os
import glob
import scanpy as sc
import matplotlib.pyplot as plt

mtx_dir = str(data_path('meta_added_h5ad'))
base_out_dir = str(data_path('scdownstream/1.doublet_detection/1.1_qcraw'))

h5ad_files = sorted(glob.glob(os.path.join(mtx_dir, "*", "*_emptydrops_filter_matrix.h5ad")))
print(f"Found {len(h5ad_files)} sample(s)")

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_emptydrops_filter_matrix.h5ad", "")
    out_dir = os.path.join(base_out_dir, name)
    os.makedirs(out_dir, exist_ok=True)

    adata_filtered = sc.read_h5ad(h5ad_path)
    sc.pp.calculate_qc_metrics(adata_filtered, inplace=True)

    sc.pl.scatter(adata_filtered, x="total_counts", y="n_genes_by_counts", show=False)
    plt.savefig(os.path.join(out_dir, f"{name}_filtered_counts_vs_genes.png"), dpi=150, bbox_inches="tight")
    plt.close()

    with open(os.path.join(out_dir, f"{name}_convert_filtered_size.txt"), "w") as fh:
        fh.write(str(adata_filtered.n_obs))

    print(f"  {name}: {adata_filtered.n_obs} cells, {adata_filtered.n_vars} genes")

print("Step 1.1 QC Raw — done.")


### Step 1.2 Scanpy Scrublet — Doublet Detection

In [ ]:
scrublet_dir = str(data_path('scdownstream/1.doublet_detection/1.2_scanpy_scrublet'))
os.makedirs(scrublet_dir, exist_ok=True)

h5ad_files = sorted(glob.glob(os.path.join(mtx_dir, "*", "*_emptydrops_filter_matrix.h5ad")))

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_emptydrops_filter_matrix.h5ad", "")
    prefix = f"{name}_scrublet"

    adata = sc.read_h5ad(h5ad_path)
    sc.pp.scrublet(adata)

    adata.obs["predicted_doublet"] = adata.obs["predicted_doublet"].astype(bool)
    df = adata.obs[["predicted_doublet"]].copy()
    df.columns = [prefix]
    df.to_pickle(os.path.join(scrublet_dir, f"{prefix}.pkl"))

    n_doublets = adata.obs["predicted_doublet"].sum()
    print(f"  {name}: {n_doublets}/{adata.n_obs} doublets ({n_doublets/adata.n_obs*100:.1f}%)")

    adata = adata[~adata.obs["predicted_doublet"]].copy()
    adata.write_h5ad(os.path.join(scrublet_dir, f"{prefix}.h5ad"))
    print(f"  {name}: {adata.n_obs} cells after doublet removal")

print("Step 1.2 Scanpy Scrublet — done.")


### Step 1.3 Doublet Removal — Merge and Filter

In [ ]:
import anndata as ad
import pandas as pd
import warnings
import upsetplot
from upsetplot import from_memberships

base_doublet_removed_dir = str(data_path('scdownstream/1.doublet_detection/1.3_doublet_remove'))
threshold = 1

def load_pred(path):
    if path.endswith(".pkl"):
        return pd.read_pickle(path)
    if path.endswith(".csv"):
        return pd.read_csv(path, index_col=0)
    raise ValueError(f"Unsupported file type: {path}")

h5ad_files = sorted(glob.glob(os.path.join(mtx_dir, "*", "*_emptydrops_filter_matrix.h5ad")))

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_emptydrops_filter_matrix.h5ad", "")
    doublet_removed_dir = os.path.join(base_doublet_removed_dir, name)
    os.makedirs(doublet_removed_dir, exist_ok=True)

    adata = ad.read_h5ad(h5ad_path)
    scrublet_pkl = os.path.join(scrublet_dir, f"{name}_scrublet.pkl")
    predictions = pd.concat([load_pred(f) for f in scrublet_pkl.split()], axis=1)
    predictions = predictions.reindex(adata.obs_names)
    mask = predictions.sum(axis=1) >= threshold
    adata = adata[~mask, :]
    adata.write_h5ad(os.path.join(doublet_removed_dir, f"{name}_doublet_rm.h5ad"))

    contents = {col: predictions[col][predictions[col].astype(bool)].index.tolist() for col in predictions.columns}
    plot_data = from_memberships(contents.values(), data=None)
    fig = plt.figure()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        upsetplot.plot(plot_data, fig=fig)
    plt.savefig(os.path.join(doublet_removed_dir, f"{name}_predictions_mqc.png"), dpi=150, bbox_inches="tight")
    plt.close()

    with open(os.path.join(doublet_removed_dir, f"{name}_doublet_removed_size.txt"), "w") as f:
        f.write(str(adata.n_obs))

    print(f"  {name}: {mask.sum()} removed, {adata.n_obs} remaining")

print("Step 1.3 Doublet Removal — done.")


### Step 2.1 QC Filter — Mitochondrial and Gene/Cell Thresholds

In [ ]:
from threadpoolctl import threadpool_limits

pct_mt_max   = 10
min_counts   = 0
min_genes    = 200
min_gene_counts = 0
min_cells    = 0
n_jobs       = 2

base_qc_filter_dir = str(data_path('scdownstream/2.qc_filter/2.1_qc_filter'))

h5ad_files = sorted(glob.glob(os.path.join(base_doublet_removed_dir, "*", "*_doublet_rm.h5ad")))
print(f"Found {len(h5ad_files)} sample(s)")

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_doublet_rm.h5ad", "")
    prefix = f"{name}_scanpy_filtered"
    qc_filter_dir = os.path.join(base_qc_filter_dir, name)
    os.makedirs(qc_filter_dir, exist_ok=True)

    threadpool_limits(n_jobs)
    sc.settings.n_jobs = n_jobs
    adata = sc.read_h5ad(h5ad_path)

    adata.var["mt"] = adata.var_names.str.lower().str.startswith("mt-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

    n_before = adata.n_obs
    adata = adata[adata.obs.pct_counts_mt < pct_mt_max, :].copy()
    sc.pp.filter_cells(adata, min_counts=min_counts)
    sc.pp.filter_genes(adata, min_counts=min_gene_counts)
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)

    adata.write_h5ad(os.path.join(qc_filter_dir, f"{prefix}.h5ad"))
    with open(os.path.join(qc_filter_dir, f"{prefix}_thresholded_size.txt"), "w") as fh:
        fh.write(str(adata.n_obs))

    print(f"  {name}: {n_before} -> {adata.n_obs} cells, {adata.n_vars} genes")

print("Step 2.1 QC Filter — done.")


### Step 2.2 FINALIZE_QC_ANNDATAS (ADATA_EXTEND)

In [ ]:
import anndata as ad

finalize_qc_dir = str(data_path('scdownstream/2.qc_filter/2.2_finalize_qc'))

h5ad_files = sorted(glob.glob(os.path.join(base_qc_filter_dir, "*", "*_scanpy_filtered.h5ad")))
print(f"Found {len(h5ad_files)} sample(s)")

for h5ad_path in h5ad_files:
    name = os.path.basename(h5ad_path).replace("_scanpy_filtered.h5ad", "")
    out_dir = os.path.join(finalize_qc_dir, name)
    os.makedirs(out_dir, exist_ok=True)

    adata = ad.read_h5ad(h5ad_path)
    # No additional obs/var/obsm/obsp/uns/layers in this run
    adata.write_h5ad(os.path.join(out_dir, f"{name}.h5ad"))
    adata.obs.to_csv(os.path.join(out_dir, f"{name}_metadata.csv"))
    print(f"  {name}: {adata.n_obs} cells, {adata.n_vars} genes")

print("Step 2.2 FINALIZE_QC_ANNDATAS — done.")


### Step 3.1. UNIFY: ADATA_UNIFY

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

# Parameters (from nextflow.config / subworkflow defaults)
batch_col             = "batch"
label_col             = ""         # no cell-type labels
unknown_label         = "unknown"
symbol_col            = "index"    # gene symbols already in var_names
counts_layer          = "X"
duplicate_var_resolution = "sum"
aggregate_isoforms    = False

unify_dir = str(data_path('scdownstream/3.unify'))
os.makedirs(unify_dir, exist_ok=True)

def aggregate_duplicate_var(adata, aggr_fun=np.mean):
    retain_var = ~adata.var_names.duplicated(keep="first")
    duplicated_var = adata.var_names[adata.var_names.duplicated()].unique()
    if len(duplicated_var):
        for var in duplicated_var:
            mask = adata.var_names == var
            var_aggr = aggr_fun(adata.X[:, mask], axis=1)[:, np.newaxis]
            adata.X[:, mask] = np.repeat(var_aggr, np.sum(mask), axis=1)
        return adata[:, retain_var].copy()
    return adata

def to_Florent_case(s):
    corrected = s.lower().strip()
    if corrected in ["na", "nan", "null", "unknown"]:
        return "unknown"
    corrected = s.replace(" ", "_").replace("-", "_")
    corrected = "".join([c if c.isalnum() or c == "_" else "" for c in corrected])
    corrected = corrected.replace("__", "_")
    if corrected.endswith("s"):
        corrected = corrected[:-1]
    corrected = corrected.strip(" _")
    return corrected.capitalize() if corrected else "unknown"

h5ad_files = sorted(glob.glob(os.path.join(finalize_qc_dir, "*", "*.h5ad")))
unified_paths = {}

for h5ad_path in h5ad_files:
    name = os.path.basename(os.path.dirname(h5ad_path))  # sample folder name

    adata = ad.read_h5ad(h5ad_path)

    # Use raw counts from X
    if counts_layer != "X":
        adata.X = adata.layers[counts_layer]

    # Strip computed slots
    adata.obsm = {}
    adata.varm = {}
    adata.uns  = {}
    adata.layers = {}

    # Convert to float32 CSR
    adata.X = csr_matrix(adata.X.astype(np.float32))

    # Aggregate isoforms (disabled)
    if aggregate_isoforms:
        adata.var_names = adata.var_names.str.replace(r'\.\d+', '', regex=True)

    # Resolve duplicate var names
    if duplicate_var_resolution in ["mean", "sum", "max"]:
        adata = aggregate_duplicate_var(adata, aggr_fun=getattr(np, duplicate_var_resolution))
    elif duplicate_var_resolution == "make_unique":
        adata.var_names_make_unique()

    # Unique obs names, prefixed with sample id
    adata.obs_names_make_unique()
    adata.obs_names = name + "_" + adata.obs_names

    # Batch column (already "batch" in obs)
    if batch_col not in adata.obs:
        adata.obs[batch_col] = name
    # batch_col == "batch" so no rename needed

    adata.obs["batch"] = adata.obs["batch"].astype(str).astype("category")

    # Label column
    if label_col:
        if label_col != "label":
            adata.obs["label"] = adata.obs[label_col]
            del adata.obs[label_col]
        if unknown_label != "unknown":
            adata.obs["label"].replace({unknown_label: "unknown"}, inplace=True)
        adata.obs["label"] = adata.obs["label"].astype(str).fillna("unknown")
        adata.obs["label"] = adata.obs["label"].map(to_Florent_case)
    else:
        adata.obs["label"] = "unknown"
    adata.obs["label"] = adata.obs["label"].astype("category")

    # Sample column
    adata.obs["sample"] = name
    adata.obs["sample"] = adata.obs["sample"].astype("category")

    # Suffix batch with sample to avoid overlap
    adata.obs["batch"] = (adata.obs["batch"].astype(str) + "_" + adata.obs["sample"].astype(str)).astype("category")

    out_path = os.path.join(unify_dir, f"{name}_unified.h5ad")
    adata.write_h5ad(out_path)
    unified_paths[name] = out_path
    print(f"  {name}: {adata.n_obs} cells, {adata.n_vars} genes")

print("UNIFY:ADATA_UNIFY — done.")


### Step 3.2. UNIFY: UPSET_GENES

In [ ]:
import warnings
import upsetplot

upset_genes_dir = os.path.join(unify_dir, "upset_genes")
os.makedirs(upset_genes_dir, exist_ok=True)

adata_genes = {
    name: ad.read_h5ad(path, backed="r").var.index.unique().tolist()
    for name, path in unified_paths.items()
}

if len(adata_genes) >= 2:
    plot_data = upsetplot.from_contents(adata_genes)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        upsetplot.plot(plot_data, sort_by="cardinality", show_counts=True, min_subset_size=10)
    plot_path = os.path.join(upset_genes_dir, "upset_genes.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {plot_path}")
else:
    print("  Skipped: need at least 2 samples for UpSet plot")

print("UNIFY:UPSET_GENES — done.")


### Step 4. COMBINE: ADATA_MERGE

In [ ]:
import scipy
import pickle
from collections import defaultdict

combine_dir = str(data_path('scdownstream/4.combine'))
os.makedirs(combine_dir, exist_ok=True)

prefix = "merged"
force_obs_cols = ""  # from nextflow.config default

# Load all unified h5ads (no base adata in this run)
adatas = [sc.read_h5ad(p) for p in sorted(unified_paths.values())]
genes = [adata.var_names for adata in adatas]

# Determine obs column intersection (+ any forced columns)
obs_col_intersection = set(adatas[0].obs.columns).intersection(
    *[adata.obs.columns for adata in adatas[1:]]
)
if force_obs_cols:
    obs_col_intersection = obs_col_intersection.union(force_obs_cols.split(","))
obs_col_intersection = sorted(obs_col_intersection)

# Collect column dtype info across all adatas
column_dtypes = defaultdict(set)
for adata in adatas:
    for dtype in ["object", "category", "number"]:
        for col in adata.obs.select_dtypes(include=dtype).columns:
            column_dtypes[col].add(dtype)

column_defaults = {}
for col, dtypes in column_dtypes.items():
    if len(dtypes) > 1:
        for adata in adatas:
            if col in adata.obs.columns:
                adata.obs[col] = adata.obs[col].astype("object")
        column_defaults[col] = "unknown"
    else:
        column_defaults[col] = np.nan if dtypes.copy().pop() == "number" else "unknown"

# Align obs columns
for adata in adatas:
    for col in set(obs_col_intersection).difference(adata.obs.columns):
        adata.obs[col] = column_defaults[col]
    adata.obs = adata.obs[obs_col_intersection]
    for col in obs_col_intersection:
        if column_dtypes[col] == {"number"}:
            continue
        adata.obs[col] = adata.obs[col].astype(str).astype("category")

# Outer concat
adata_outer = ad.concat(adatas, join="outer")
adata_outer.X = csr_matrix(adata_outer.X)
adata_outer.obs = adata_outer.obs.reindex(sorted(adata_outer.obs.columns), axis=1)

# Inner (gene intersection)
gene_intersection = set(genes[0]).intersection(*genes[1:])
intersection_mask = adata_outer.var_names.to_series(name="intersection").map(lambda x: x in gene_intersection)
adata_inner = adata_outer[:, intersection_mask].copy()

# Save
adata_outer.write_h5ad(os.path.join(combine_dir, f"{prefix}_outer.h5ad"))
adata_inner.write_h5ad(os.path.join(combine_dir, f"{prefix}_inner.h5ad"))
# No base adata → integrate = inner (symlink equivalent: just copy path)
adata_inner.write_h5ad(os.path.join(combine_dir, f"{prefix}_integrate.h5ad"))
intersection_mask.to_pickle(os.path.join(combine_dir, "gene_intersection.pkl"))

print(f"  Outer  : {adata_outer.n_obs} cells x {adata_outer.n_vars} genes")
print(f"  Inner  : {adata_inner.n_obs} cells x {adata_inner.n_vars} genes (shared genes)")
print(f"  Output : {combine_dir}")
print("COMBINE:ADATA_MERGE — done.")


### Step 5.0 — Cell Type Annotation with CellTypist
Annotate cells using the Immune All Low atlas model (Immune_All_Low.pkl) before HVG selection.

> Input: merged_integrate.h5ad from Step 4 combine (4.combine/merged_integrate.h5ad). <br>
> Model: test_data/atlas/Immune_All_Low.pkl <br>
> Output: merged_integrate_celltypist.h5ad saved under 5.pre_hvg_annotation/ with celltypist:* columns added to adata.obs. <br>

Note on gene names: CellTypist requires gene symbols as var_names. If your var_names are
Ensembl IDs, uncomment the adata_ct.var_names = ... line and set symbol_col to the correct column.

In [ ]:
import os
import pandas as pd
import scanpy as sc
import celltypist
from celltypist import models as ct_models
import mygene

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad  = str(data_path('output/merged_integrate.h5ad'))
model_path  = str(atlas_path()) #### please change to your local model path
annot_dir   = str(data_path('output/5.0_pre_hvg_annotation'))
out_h5ad    = os.path.join(annot_dir, "merged_celltypist_immune_low.h5ad")
out_obs_pkl = os.path.join(annot_dir, "celltypist_obs.pkl")
os.makedirs(annot_dir, exist_ok=True)

# ── Load AnnData ───────────────────────────────────────────────────────────────
adata = sc.read_h5ad(input_h5ad)
print(f"Loaded: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"var_names (first 5): {list(adata.var_names[:5])}")

# ── Prepare a copy for CellTypist (requires normalised log counts) ─────────────
adata_ct = adata.copy()
sc.pp.normalize_per_cell(adata_ct, counts_per_cell_after=1e4)
sc.pp.log1p(adata_ct)

# If var_names are Ensembl IDs, switch to gene symbols before annotating.
# Uncomment and set symbol_col to the relevant column in adata.var (e.g. "gene_symbols").
# symbol_col = "gene_symbols"
# adata_ct.var_names = adata_ct.var[symbol_col]

# ── Load local model ───────────────────────────────────────────────────────────
model_name = os.path.basename(model_path)[:-4]   # "Mouse_Whole_Brain"
model_obj  = ct_models.Model.load(model_path)
print(f"Model loaded: {model_name}")
print(f"Model cell types: {len(model_obj.cell_types)}")

# ── Run annotation ─────────────────────────────────────────────────────────────
# Make sure the annotation object uses gene symbols, not Ensembl IDs.
# CellTypist requires `var_names` to match model features.
symbol_col = None
for c in ("gene_symbols", "symbol", "gene_symbol", "Symbol", "GeneSymbol"):
    if c in adata_ct.var.columns:
        symbol_col = c
        break

if symbol_col is not None:
    adata_ct.var_names = adata_ct.var[symbol_col].astype(str)
else:
    adata_ct.var_names = adata_ct.var_names.astype(str)

# Keep only genes present in the loaded CellTypist model.
model_features = set(model_obj.features)
overlap_mask = [g in model_features for g in adata_ct.var_names]
if not any(overlap_mask):
    # CellTypist features are stored as gene symbols; many AnnData objects use
    # a different case or a symbol column. Try a case-insensitive alignment first.
    model_features_uc = {str(g).upper() for g in model_obj.features}
    var_names_uc = adata_ct.var_names.astype(str).str.upper()

    overlap_mask = [g in model_features_uc for g in var_names_uc]
    if any(overlap_mask):
        adata_ct.var_names = var_names_uc
        adata_ct = adata_ct[:, overlap_mask].copy()
    else:
        # If a dedicated symbol column exists, use it explicitly.
        if symbol_col is not None:
            adata_ct.var_names = adata_ct.var[symbol_col].astype(str).str.upper()
            overlap_mask = [g in model_features_uc for g in adata_ct.var_names]
            if any(overlap_mask):
                adata_ct = adata_ct[:, overlap_mask].copy()
            else:
                raise ValueError(
                    "No features overlap with the model after symbol remapping. "
                    "Please provide gene symbols in `adata_ct.var_names` or in a "
                    "`gene_symbols`/`symbol` column that matches the CellTypist model features."
                )
        else:
            # If the data are still using Ensembl IDs, try an Ensembl -> gene-symbol lookup.
            # This is the common case when `adata.var_names` are `ENSG...` IDs.
            if all(str(g).startswith("ENSG") for g in adata_ct.var_names[: min(100, len(adata_ct.var_names))]):
                try:
                    mg = mygene.MyGeneInfo()

                    mapping = mg.querymany(
                        list(adata_ct.var_names),
                        scopes="ensembl.gene",
                        fields="symbol",
                        species="human",
                        returnall=False,
                        verbose=False,
                    )

                    symbol_map = {}
                    for item in mapping:
                        q = item.get("query")
                        if q is None:
                            continue
                        symbol = item.get("symbol")
                        if symbol:
                            symbol_map[q] = symbol

                    if symbol_map:
                        adata_ct.var_names = pd.Index(
                            [symbol_map.get(str(g), str(g)) for g in adata_ct.var_names]
                        )
                        model_features_uc = {str(g).upper() for g in model_obj.features}
                        var_names_uc = adata_ct.var_names.astype(str).str.upper()

                        overlap_mask = [g in model_features_uc for g in var_names_uc]
                        if any(overlap_mask):
                            adata_ct = adata_ct[:, overlap_mask].copy()
                        else:
                            raise ValueError(
                                "No features overlap with the model after Ensembl-to-symbol mapping. "
                                "Please provide gene symbols in `adata_ct.var_names` or in a "
                                "`gene_symbols`/`symbol` column that matches the CellTypist model features."
                            )
                    else:
                        raise ValueError(
                            "No features overlap with the model. "
                            "Your `adata_ct.var_names` appear to be Ensembl IDs, but no gene-symbol "
                            "mapping was found. Please provide gene symbols in `adata_ct.var_names` "
                            "or in a `gene_symbols`/`symbol` column."
                        )
                except Exception:
                    raise ValueError(
                        "No features overlap with the model. "
                        "Your `adata_ct.var_names` do not match the CellTypist gene symbols "
                        "(case-insensitive overlap was also unsuccessful, and Ensembl-to-symbol "
                        "mapping was not available). Please provide gene symbols in "
                        "`adata_ct.var_names` or in a `gene_symbols`/`symbol` column."
                    ) from None
            else:
                raise ValueError(
                    "No features overlap with the model. "
                    "Your `adata_ct.var_names` do not match the CellTypist gene symbols "
                    "(case-insensitive overlap was also unsuccessful). "
                    "Please provide gene symbols in `adata_ct.var_names` or in a "
                    "`gene_symbols`/`symbol` column."
                )

# Rebuild the overlap mask against the current adata_ct shape (5782 vars),
# instead of reusing the stale mask that was computed before symbol remapping.
current_var_names = adata_ct.var_names.astype(str)
overlap_mask = [g in model_features for g in current_var_names]

if not any(overlap_mask):
    model_features_uc = {str(g).upper() for g in model_obj.features}
    overlap_mask = [str(g).upper() in model_features_uc for g in current_var_names]

if not any(overlap_mask):
    raise ValueError(
        "No features overlap with the model after current name normalization."
    )

adata_ct = adata_ct[:, overlap_mask].copy()

# Run annotation on the overlap-only object.
predictions = celltypist.annotate(
    adata_ct,
    model=model_obj,
    majority_voting=True,
)
predictions_adata = predictions.to_adata()

df_ct = predictions_adata.obs.loc[
    adata.obs.index, ["predicted_labels", "conf_score"]
].copy()
df_ct.columns = [f"celltypist:{model_name}", f"celltypist:{model_name}:conf"]

# ── Merge results back into original adata ─────────────────────────────────────
adata.obs = pd.concat([adata.obs, df_ct], axis=1)

# ── Save ───────────────────────────────────────────────────────────────────────
df_ct.to_pickle(out_obs_pkl)
adata.write_h5ad(out_h5ad)

print(f"\nAnnotation complete.")
print(f"  Unique cell types assigned : {adata.obs[f'celltypist:{model_name}'].nunique()}")
print(f"  Output h5ad  : {out_h5ad}")
print(f"  Output obs   : {out_obs_pkl}")

### Step 5.1 Scanpy HVGs — Combined Dataset

In [ ]:
import os
import scanpy as sc
import numpy as np

# ── Parameters ─────────────────────────────────────────────────────────────────
n_hvgs    = 3000   # number of top HVGs (set to 0 to skip subsetting)
batch_key = "batch"  # batch-aware HVG selection across samples

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad = str(data_path('output/5.0_pre_hvg_annotation/merged_celltypist_immune_low.h5ad'))
hvgs_dir   = str(data_path('output/5.1_merged_hvgs'))
os.makedirs(hvgs_dir, exist_ok=True)

prefix = "merged_hvgs"

adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells x {adata.n_vars} genes")

# Preserve raw counts before log-normalisation
raw_counts = adata.X.copy()

# Log-normalise for HVG scoring
sc.pp.log1p(adata)

# Batch-aware HVG selection
kwargs = {"n_top_genes": n_hvgs}
if batch_key and batch_key in adata.obs.columns:
    kwargs["batch_key"] = batch_key

sc.pp.highly_variable_genes(adata, **kwargs)

# Save HVG boolean flags as pickle
adata.var[["highly_variable"]].to_pickle(os.path.join(hvgs_dir, f"{prefix}.pkl"))

# Restore raw counts
adata.X = raw_counts

# Save full-gene matrix (all genes, raw counts)
adata.write_h5ad(os.path.join(hvgs_dir, f"{prefix}_full.h5ad"))

# Subset to HVGs and save
n_total = adata.n_vars
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
adata_hvg.write_h5ad(os.path.join(hvgs_dir, f"{prefix}.h5ad"))

print(f"  HVGs selected : {adata_hvg.n_vars} / {n_total}")
print(f"  Output: {hvgs_dir}")
print("Step 5.1 Scanpy HVGs — done.")


In [ ]:
import pandas as pd
# As a numpy array (cells x HVGs)
raw = adata_hvg.X.toarray()

# As a DataFrame (rows=cells, columns=gene names)
df = pd.DataFrame(raw, index=adata_hvg.obs_names, columns=adata_hvg.var_names)
df


### Step 5.2.1 NB-VAE — Custom Variational Autoencoder with Negative Binomial Likelihood

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.sparse import issparse
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# ── Parameters ─────────────────────────────────────────────────────────────────
n_latent   = 30
n_hidden   = [256, 128]
n_epochs   = 800
batch_size = 128
lr         = 1e-4
kl_warmup  = 50       # epochs to linearly ramp KL weight from 0 → 1
patience   = 30       # early stopping patience (epochs)
val_frac   = 0.1      # fraction of cells for validation
batch_key  = "batch"  # obs column for batch correction (set "" to disable)

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad = str(data_path('output/5.1_merged_hvgs/merged_hvgs.h5ad'))
out_dir    = str(data_path('output/5.2.1_NBVAE'))
os.makedirs(out_dir, exist_ok=True)
prefix = "merged_nbvae"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device : {device}")


# ── Model ──────────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, n_input, n_hidden, n_latent, n_batch=0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input + n_batch, n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
            nn.Linear(n_hidden[0], n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
        )
        self.fc_mu      = nn.Linear(n_hidden[1], n_latent)
        self.fc_log_var = nn.Linear(n_hidden[1], n_latent)

    def forward(self, x, batch_oh=None):
        inp = torch.cat([x, batch_oh], dim=1) if batch_oh is not None else x
        h = self.net(inp)
        return self.fc_mu(h), self.fc_log_var(h)


class Decoder(nn.Module):
    def __init__(self, n_latent, n_hidden, n_output, n_batch=0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_latent + n_batch, n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
            nn.Linear(n_hidden[1], n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
        )
        self.fc_scale  = nn.Linear(n_hidden[0], n_output)
        # Per-gene log dispersion (shared across cells, learned during training)
        self.log_theta = nn.Parameter(torch.zeros(n_output))

    def forward(self, z, log_library, batch_oh=None):
        inp = torch.cat([z, batch_oh], dim=1) if batch_oh is not None else z
        h = self.net(inp)
        # Mean: softmax(scale) * library_size  → each gene's expected count
        px_scale = F.softmax(self.fc_scale(h), dim=-1)
        px_mu    = px_scale * torch.exp(log_library).unsqueeze(1)
        px_theta = torch.exp(self.log_theta)
        return px_mu, px_theta


class NBVAE(nn.Module):
    def __init__(self, n_genes, n_hidden, n_latent, n_batch=0):
        super().__init__()
        self.encoder = Encoder(n_genes, n_hidden, n_latent, n_batch)
        self.decoder = Decoder(n_latent, n_hidden, n_genes, n_batch)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + torch.randn_like(std) * std

    def forward(self, x, log_library, batch_oh=None):
        mu_z, log_var_z = self.encoder(torch.log1p(x), batch_oh)
        z = self.reparameterize(mu_z, log_var_z)
        px_mu, px_theta = self.decoder(z, log_library, batch_oh)
        return px_mu, px_theta, mu_z, log_var_z

    @torch.no_grad()
    def get_latent(self, x, log_library, batch_oh=None):
        self.eval()
        mu_z, _ = self.encoder(torch.log1p(x), batch_oh)
        return mu_z


# ── Loss ───────────────────────────────────────────────────────────────────────
def nb_log_likelihood(x, mu, theta, eps=1e-8):
    log_theta_mu = torch.log(theta + mu + eps)
    return (
        theta * (torch.log(theta + eps) - log_theta_mu)
        + x   * (torch.log(mu    + eps) - log_theta_mu)
        + torch.lgamma(x + theta)
        - torch.lgamma(theta)
        - torch.lgamma(x + 1)
    )

def elbo(x, px_mu, px_theta, mu_z, log_var_z, kl_weight):
    recon = nb_log_likelihood(x, px_mu, px_theta).sum(dim=1).mean()
    kl    = -0.5 * (1 + log_var_z - mu_z.pow(2) - log_var_z.exp()).sum(dim=1).mean()
    return -recon + kl_weight * kl, recon.item(), kl.item()


# ── Data preparation ───────────────────────────────────────────────────────────
adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells x {adata.n_vars} genes")

X = adata.X.toarray().astype(np.float32) if issparse(adata.X) else adata.X.astype(np.float32)
lib_size = X.sum(axis=1, keepdims=True).clip(1)
log_lib  = np.log(lib_size).squeeze(1)

# Batch one-hot encoding
n_batch_cats = 0
batch_oh_np  = None
if batch_key and batch_key in adata.obs.columns:
    cats = adata.obs[batch_key].astype("category")
    n_batch_cats = len(cats.cat.categories)
    batch_oh_np  = np.eye(n_batch_cats, dtype=np.float32)[cats.cat.codes.values]
    print(f"  Batches: {list(cats.cat.categories)}")

# Train / val split
n_cells  = X.shape[0]
n_val    = max(1, int(n_cells * val_frac))
idx      = np.random.permutation(n_cells)
val_idx, train_idx = idx[:n_val], idx[n_val:]

def make_tensors(indices):
    x_t   = torch.tensor(X[indices]).to(device)
    ll_t  = torch.tensor(log_lib[indices]).to(device)
    oh_t  = torch.tensor(batch_oh_np[indices]).to(device) if batch_oh_np is not None else None
    return TensorDataset(x_t, ll_t) if oh_t is None else TensorDataset(x_t, ll_t, oh_t)

train_loader = DataLoader(make_tensors(train_idx), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(make_tensors(val_idx),   batch_size=batch_size)

# ── Training ───────────────────────────────────────────────────────────────────
model = NBVAE(adata.n_vars, n_hidden, n_latent, n_batch_cats).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=lr)

train_losses, val_losses = [], []
best_val, no_improve, best_state = np.inf, 0, None

for epoch in range(1, n_epochs + 1):
    kl_w = min(1.0, epoch / kl_warmup)

    model.train()
    t_loss = 0
    for batch in train_loader:
        x_b, ll_b = batch[0], batch[1]
        oh_b = batch[2] if len(batch) == 3 else None
        px_mu, px_theta, mu_z, lv_z = model(x_b, ll_b, oh_b)
        loss, _, _ = elbo(x_b, px_mu, px_theta, mu_z, lv_z, kl_w)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        t_loss += loss.item() * x_b.size(0)
    train_losses.append(t_loss / len(train_idx))

    model.eval()
    v_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            x_b, ll_b = batch[0], batch[1]
            oh_b = batch[2] if len(batch) == 3 else None
            px_mu, px_theta, mu_z, lv_z = model(x_b, ll_b, oh_b)
            loss, _, _ = elbo(x_b, px_mu, px_theta, mu_z, lv_z, kl_weight=1.0)
            v_loss += loss.item() * x_b.size(0)
    val_losses.append(v_loss / len(val_idx))

    if val_losses[-1] < best_val:
        best_val, no_improve = val_losses[-1], 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        no_improve += 1

    if epoch % 50 == 0:
        print(f"  Epoch {epoch:4d} | train {train_losses[-1]:.2f} | val {val_losses[-1]:.2f} | kl_w {kl_w:.2f}")

    if no_improve >= patience:
        print(f"  Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"  Best val loss: {best_val:.4f}")

# ── Loss curve ─────────────────────────────────────────────────────────────────
plt.figure(figsize=(6, 3))
plt.plot(train_losses, label="train")
plt.plot(val_losses,   label="val")
plt.xlabel("Epoch"); plt.ylabel("ELBO loss"); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(out_dir, f"{prefix}_loss.png"), dpi=150)
plt.show(); plt.close()

# ── Extract latent embedding ───────────────────────────────────────────────────
model.eval()
all_x   = torch.tensor(X).to(device)
all_ll  = torch.tensor(log_lib).to(device)
all_oh  = torch.tensor(batch_oh_np).to(device) if batch_oh_np is not None else None

with torch.no_grad():
    Z = model.get_latent(all_x, all_ll, all_oh).cpu().numpy()

adata.obsm["X_emb"] = Z

# ── Save ───────────────────────────────────────────────────────────────────────
adata.write_h5ad(os.path.join(out_dir, f"{prefix}.h5ad"))
pd.DataFrame(Z, index=adata.obs_names).to_pickle(os.path.join(out_dir, f"X_{prefix}.pkl"))
torch.save(best_state, os.path.join(out_dir, f"{prefix}_model.pt"))

print(f"  Latent shape : {Z.shape}")
print(f"  Output       : {out_dir}")
print("Step 5.2 NB-VAE — done.")


### Step 5.2.2 PCA + Harmony — Batch-Corrected Baseline Embedding

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import harmonypy as hm
import matplotlib.pyplot as plt

# ── Parameters ─────────────────────────────────────────────────────────────────
n_pcs     = 30
batch_key = "batch"

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad = str(data_path('output/5.1_merged_hvgs/merged_hvgs.h5ad'))
out_dir    = str(data_path('output/5.2.2_PCA_Harmony'))
os.makedirs(out_dir, exist_ok=True)
prefix = "merged_harmony"

adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells x {adata.n_vars} genes")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*log-transformed.*")
    warnings.filterwarnings("ignore", message=".*zero-centering.*")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.scale(adata, max_value=10)

sc.pp.pca(adata, n_comps=n_pcs)
print(f"  PCA done : {n_pcs} components")

# ── UMAP before Harmony (raw PCA) ─────────────────────────────────────────────
sc.pp.neighbors(adata, use_rep="X_pca", n_pcs=n_pcs)
sc.tl.umap(adata, random_state=0)
adata.obsm["X_umap_before"] = adata.obsm["X_umap"].copy()
print("  UMAP before Harmony computed")

# ── Run Harmony ────────────────────────────────────────────────────────────────
x = adata.obsm["X_pca"].astype(np.float64)
ho = hm.run_harmony(x, adata.obs, batch_key)

Z = np.array(ho.Z_corr)
if Z.shape[0] == n_pcs and Z.shape[1] == adata.n_obs:
    Z = Z.T

adata.obsm["X_emb"] = Z
print(f"  Harmony done — X_emb shape: {adata.obsm['X_emb'].shape}")

# ── UMAP after Harmony (corrected embedding) ───────────────────────────────────
sc.pp.neighbors(adata, use_rep="X_emb", n_pcs=n_pcs)
sc.tl.umap(adata, random_state=0)
adata.obsm["X_umap_after"] = adata.obsm["X_umap"].copy()
print("  UMAP after Harmony computed")

# ── Verification plot: before vs after batch correction ────────────────────────
# For Harmony there is no training loss curve (unlike NB-VAE).
# Instead, we verify batch correction by comparing UMAP colored by batch:
#   Left  = before Harmony: batches may cluster separately  → correction needed
#   Right = after  Harmony: batches should overlap          → correction worked
batches = adata.obs[batch_key].astype("category")
batch_cats = batches.cat.categories.tolist()
palette = plt.cm.tab10(np.linspace(0, 1, len(batch_cats)))
b2c = {b: palette[i] for i, b in enumerate(batch_cats)}
cell_colors = [b2c[b] for b in batches]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, key, title in [
    (axes[0], "X_umap_before", "Before Harmony\n(raw PCA — batches may separate)"),
    (axes[1], "X_umap_after",  "After Harmony\n(corrected — batches should overlap)"),
]:
    coords = adata.obsm[key]
    ax.scatter(coords[:, 0], coords[:, 1], c=cell_colors, s=1, alpha=0.4, rasterized=True)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

# shared legend
handles = [plt.Line2D([0],[0], marker='o', color='w',
                      markerfacecolor=b2c[b], markersize=7, label=b) for b in batch_cats]
fig.legend(handles=handles, title=batch_key, loc="lower center",
           ncol=len(batch_cats), fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.02))
plt.suptitle("Batch correction verification — PCA + Harmony", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, f"{prefix}_batch_correction.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print(f"  Saved : {prefix}_batch_correction.png")

# ── Save outputs ───────────────────────────────────────────────────────────────
adata.write_h5ad(os.path.join(out_dir, f"{prefix}.h5ad"))
pd.DataFrame(adata.obsm["X_emb"], index=adata.obs_names).to_pickle(
    os.path.join(out_dir, f"X_{prefix}.pkl")
)

print(f"  Output : {out_dir}")
print("Step 5.3 PCA + Harmony — done.")

#### How to read the Step 5.2.2 batch correction plot

| | Before Harmony | After Harmony |
|---|---|---|
| **What you see** | Batches form somewhat separate regions | Batches are well mixed across clusters |
| **What it means** | Raw PCA captures batch variance alongside biology | Harmony has removed batch effects from the embedding |

**Key differences from Step 5.2 (NB-VAE):**
- Step 5.2.1 verifies **model convergence** via a train vs validation ELBO loss curve — the val loss should decrease and plateau without diverging from train loss.
- Step 5.2.2 verifies **batch correction quality** via a before/after UMAP — there is no training loss because Harmony is not a neural network; it is an iterative algorithm that adjusts PC coordinates to minimize batch-driven clustering.

**What good Harmony correction looks like:**
- Batches that were spatially separated in the "Before" UMAP should overlap in the "After" UMAP.
- Biologically distinct clusters (cell types) should still remain separated after correction — Harmony should mix batches *within* clusters, not collapse all clusters together.
- Harmony logs the number of iterations until convergence (e.g., "Converged after 7 iterations"). Fewer iterations = batches were already relatively similar; more iterations = stronger batch effect needed more correction.

### Step 5.2.3 Custom PyTorch scVI-Style VAE

This custom PyTorch variational autoencoder produced the scVI-style embedding used for the manuscript-reported comparison. It is distinct from the separate scvi-tools implementation in `scripts/step5_2_3_scvi.py`.


In [ ]:
# Custom PyTorch scVI-style VAE used for the manuscript-reported comparison.

import os
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.sparse import issparse
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# ── Parameters ─────────────────────────────────────────────────────────────────
n_latent   = 30
n_hidden   = [256, 128]   
embed_dim  = 8            
n_epochs   = 800
batch_size = 128          
lr         = 1e-4
kl_warmup  = 50
patience   = 30          
val_frac   = 0.1
batch_key  = "batch"

input_h5ad = str(data_path("output/5.1_merged_hvgs/merged_hvgs.h5ad"))
out_dir    = str(data_path("output/5.2.3_scVI/scvi_custom"))
os.makedirs(out_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device : {device}")


# ── Model ──────────────────────────────────────────────────────────────────────

# BatchEmbedding replaces raw one-hot.
#   NB-VAE:  batch_oh = np.eye(K)[codes]          → fixed [1,0,0] / [0,1,0] / [0,0,1]
#   scVI:    nn.Embedding(K, embed_dim)            → learned 8-float vector per batch,
#            updated by backprop like any weight.
class BatchEmbedding(nn.Module):
    def __init__(self, n_batch, embed_dim):
        super().__init__()
        self.embed = nn.Embedding(n_batch, embed_dim)

    def forward(self, batch_idx):          # batch_idx: integer tensor (N,)
        return self.embed(batch_idx)       # → (N, embed_dim)  continuous floats


# Library size encoder.
#   NB-VAE:  log_library is computed once from raw counts and passed as a fixed number.
#   scVI:    library size l is treated as a latent variable with its own encoder
#            q(l | x, batch) = N(mu_l, exp(lv_l)), sampled each forward pass.
class LibEncoder(nn.Module):
    def __init__(self, n_input, n_hidden, embed_dim):
        super().__init__()
        self.net   = nn.Sequential(
            nn.Linear(n_input + embed_dim, n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
        )
        self.mu    = nn.Linear(n_hidden[0], 1)
        self.logv  = nn.Linear(n_hidden[0], 1)

    def forward(self, x, batch_emb):
        h = self.net(torch.cat([x, batch_emb], dim=1))
        return self.mu(h), self.logv(h)    # (N,1), (N,1)


class Encoder(nn.Module):
    def __init__(self, n_input, n_hidden, n_latent, embed_dim):
        super().__init__()
        # input size = n_genes + embed_dim  (NB-VAE: n_genes + n_batch_cats)
        self.net = nn.Sequential(
            nn.Linear(n_input + embed_dim, n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
            nn.Linear(n_hidden[0], n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
        )
        self.fc_mu      = nn.Linear(n_hidden[1], n_latent)
        self.fc_log_var = nn.Linear(n_hidden[1], n_latent)

    def forward(self, x, batch_emb):
        h = self.net(torch.cat([x, batch_emb], dim=1))
        return self.fc_mu(h), self.fc_log_var(h)


class Decoder(nn.Module):
    def __init__(self, n_latent, n_hidden, n_output, embed_dim):
        super().__init__()
        # input size = n_latent + embed_dim  (NB-VAE: n_latent + n_batch_cats)
        self.net = nn.Sequential(
            nn.Linear(n_latent + embed_dim, n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
            nn.Linear(n_hidden[0], n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
        )
        self.fc_scale   = nn.Linear(n_hidden[1], n_output)  # gene proportions (same as NB-VAE)
        self.log_theta  = nn.Parameter(torch.zeros(n_output))  # per-gene dispersion (same)
        # fc_dropout adds the ZINB zero-inflation head.
        #   NB-VAE has NO this layer — NB assumes every zero is a true biological zero.
        #   scVI adds logit(π): probability that a count is a "dropout zero" (technical).
        self.fc_dropout = nn.Linear(n_hidden[1], n_output)

    def forward(self, z, library, batch_emb):
        h = self.net(torch.cat([z, batch_emb], dim=1))
        px_scale   = F.softmax(self.fc_scale(h), dim=-1)
        px_mu      = px_scale * torch.exp(library)   # mean counts
        px_theta   = torch.exp(self.log_theta)        # dispersion
        px_dropout = self.fc_dropout(h)               # logit(π), raw (not sigmoid yet)
        return px_mu, px_theta, px_dropout


class ScVICustom(nn.Module):
    def __init__(self, n_genes, n_hidden, n_latent, n_batch, embed_dim):
        super().__init__()
        self.batch_emb = BatchEmbedding(n_batch, embed_dim)  # ← DIFF
        self.lib_enc   = LibEncoder(n_genes, n_hidden, embed_dim)  # ← DIFF
        self.encoder   = Encoder(n_genes, n_hidden, n_latent, embed_dim)
        self.decoder   = Decoder(n_latent, n_hidden, n_genes, embed_dim)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + torch.randn_like(std) * std

    def forward(self, x, batch_idx):
        b              = self.batch_emb(batch_idx)                # integer → embedding
        mu_z, lv_z     = self.encoder(torch.log1p(x), b)
        mu_l, lv_l     = self.lib_enc(torch.log1p(x), b)         # learn library size
        z              = self.reparameterize(mu_z, lv_z)
        l              = self.reparameterize(mu_l, lv_l)          # sampled log-library
        px_mu, px_theta, px_dropout = self.decoder(z, l, b)
        return px_mu, px_theta, px_dropout, mu_z, lv_z, mu_l, lv_l

    @torch.no_grad()
    def get_latent(self, x, batch_idx):
        self.eval()
        b = self.batch_emb(batch_idx)
        mu_z, _ = self.encoder(torch.log1p(x), b)
        return mu_z


# ── Loss ───────────────────────────────────────────────────────────────────────
# ZINB log-likelihood replaces NB log-likelihood.
#
#   NB-VAE (NB):
#     p(x | z) = NB(x ; mu, theta)          — no zero-inflation term
#
#   scVI (ZINB):
#     p(x | z) = pi * delta(x=0)  +  (1-pi) * NB(x ; mu, theta)
#
#     pi = sigmoid(logit_pi)  = probability this zero is a DROPOUT ZERO (technical)
#     For x = 0:  log[ pi + (1-pi)*NB(0;mu,theta) ]   <- two ways to get a zero
#     For x > 0:  log(1-pi) + log NB(x;mu,theta)       <- must not be a dropout
#
def zinb_log_likelihood(x, mu, theta, logit_pi, eps=1e-8):
    pi  = torch.sigmoid(logit_pi)                         # dropout prob
    log_theta_mu = torch.log(theta + mu + eps)
    nb_ll = (                                              # NB term (same math as NB-VAE)
        theta * (torch.log(theta + eps) - log_theta_mu)
        + x   * (torch.log(mu    + eps) - log_theta_mu)
        + torch.lgamma(x + theta)
        - torch.lgamma(theta)
        - torch.lgamma(x + 1)
    )
    # log NB(0; mu, theta) — used for x=0 branch
    nb_zero = theta * (torch.log(theta + eps) - torch.log(theta + mu + eps))
    # Numerically stable log-sum-exp for the x=0 case
    log_prob_zero = torch.logaddexp(
        torch.log(pi + eps),
        torch.log(1 - pi + eps) + nb_zero
    )
    log_prob_pos  = torch.log(1 - pi + eps) + nb_ll      # x > 0 case
    return torch.where(x < 0.5, log_prob_zero, log_prob_pos)


def elbo(x, px_mu, px_theta, px_dropout, mu_z, lv_z, mu_l, lv_l, kl_weight):
    recon  = zinb_log_likelihood(x, px_mu, px_theta, px_dropout).sum(dim=1).mean()
    kl_z   = -0.5 * (1 + lv_z - mu_z.pow(2) - lv_z.exp()).sum(dim=1).mean()
    kl_l   = -0.5 * (1 + lv_l - mu_l.pow(2) - lv_l.exp()).sum(dim=1).mean()  # ← DIFF
    return -recon + kl_weight * (kl_z + kl_l), recon.item(), kl_z.item()


# ── Data preparation ───────────────────────────────────────────────────────────
adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells x {adata.n_vars} genes")

X = adata.X.toarray().astype(np.float32) if issparse(adata.X) else adata.X.astype(np.float32)

# batch stored as INTEGER index, not one-hot matrix.
#   NB-VAE:  batch_oh_np = np.eye(K)[codes]   → shape (N, K)  float matrix
#   scVI:    batch_codes = codes               → shape (N,)    int array
cats        = adata.obs[batch_key].astype("category")
n_batch_cats = len(cats.cat.categories)
batch_codes  = cats.cat.codes.values.astype(np.int64)
print(f"  Batches: {list(cats.cat.categories)}")

n_cells  = X.shape[0]
n_val    = max(1, int(n_cells * val_frac))
idx      = np.random.permutation(n_cells)
val_idx, train_idx = idx[:n_val], idx[n_val:]

def make_tensors(indices):
    x_t  = torch.tensor(X[indices]).to(device)
    bi_t = torch.tensor(batch_codes[indices]).to(device)  # integer, not one-hot
    return TensorDataset(x_t, bi_t)

train_loader = DataLoader(make_tensors(train_idx), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(make_tensors(val_idx),   batch_size=batch_size)


# ── Training ───────────────────────────────────────────────────────────────────
model = ScVICustom(adata.n_vars, n_hidden, n_latent, n_batch_cats, embed_dim).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=lr)

train_losses, val_losses = [], []
best_val, no_improve, best_state = np.inf, 0, None

for epoch in range(1, n_epochs + 1):
    kl_w = min(1.0, epoch / kl_warmup)

    model.train()
    t_loss = 0
    for x_b, bi_b in train_loader:
        px_mu, px_theta, px_dropout, mu_z, lv_z, mu_l, lv_l = model(x_b, bi_b)
        loss, _, _ = elbo(x_b, px_mu, px_theta, px_dropout, mu_z, lv_z, mu_l, lv_l, kl_w)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        t_loss += loss.item() * x_b.size(0)
    train_losses.append(t_loss / len(train_idx))

    model.eval()
    v_loss = 0
    with torch.no_grad():
        for x_b, bi_b in val_loader:
            px_mu, px_theta, px_dropout, mu_z, lv_z, mu_l, lv_l = model(x_b, bi_b)
            loss, _, _ = elbo(x_b, px_mu, px_theta, px_dropout, mu_z, lv_z, mu_l, lv_l, kl_weight=1.0)
            v_loss += loss.item() * x_b.size(0)
    val_losses.append(v_loss / len(val_idx))

    if val_losses[-1] < best_val:
        best_val, no_improve = val_losses[-1], 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        no_improve += 1

    if epoch % 50 == 0:
        print(f"  Epoch {epoch:4d} | train {train_losses[-1]:.2f} | val {val_losses[-1]:.2f} | kl_w {kl_w:.2f}")

    if no_improve >= patience:
        print(f"  Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"  Best val loss: {best_val:.4f}")

# ── Loss curve for convergence diagnostics ──────────────────────────────────────
# Match the Step 5.2.1 NB-VAE style: train vs validation loss over epochs.
if train_losses and val_losses:
    epochs = np.arange(1, len(train_losses) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_losses, label="train", color="tab:blue", linewidth=2)
    plt.plot(epochs, val_losses, label="validation", color="tab:orange", linestyle="--", linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("ELBO loss")
    plt.title("scVI custom — train vs validation loss")
    plt.legend()
    plt.tight_layout()
    loss_path = os.path.join(out_dir, "loss_vs_epoch.png")
    plt.savefig(loss_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"  Loss plot : {loss_path}")

    pd.DataFrame({
        "epoch": epochs,
        "train_loss": train_losses,
        "val_loss": val_losses,
    }).to_csv(os.path.join(out_dir, "loss_vs_epoch.csv"), index=False)

# ── Extract latent embedding ───────────────────────────────────────────────────
model.eval()
all_x  = torch.tensor(X).to(device)
all_bi = torch.tensor(batch_codes).to(device)          # integer indices

with torch.no_grad():
    Z = model.get_latent(all_x, all_bi).cpu().numpy()

adata.obsm["X_emb"] = Z

# ── Save ───────────────────────────────────────────────────────────────────────
adata.write_h5ad(os.path.join(out_dir, "merged_scvi_custom.h5ad"))
pd.DataFrame(Z, index=adata.obs_names).to_pickle(os.path.join(out_dir, "X_scvi_custom.pkl"))
torch.save(best_state, os.path.join(out_dir, "scvi_custom_model.pt"))

print(f"  Latent shape : {Z.shape}")
print(f"  Output       : {out_dir}")
print("Step 5.2.3 custom PyTorch scVI-style VAE — done.")

### Embedding comparison — manuscript-reported results

The reported comparison uses the custom PyTorch scVI-style VAE embedding from Step 5.2.3, alongside the custom NB-VAE and PCA + Harmony embeddings. The separate scvi-tools SCVI implementation is not the source of the manuscript-reported values.

| Method | Batch ASW (↓) | Bio ASW (↑) | kNN batch purity (↓) | Batch correction (↑) | Bio preservation (↑) | Overall score (↑) |
|---|---:|---:|---:|---:|---:|---:|
| NB-VAE | 0.0035 | −0.0035 | 0.4172 | 0.9965 | 0.4982 | 0.697527 |
| PCA + Harmony | 0.0366 | −0.0366 | 0.4085 | 0.9634 | 0.4817 | 0.674383 |
| Custom PyTorch scVI-style VAE | 0.0035 | −0.0035 | 0.4345 | 0.9965 | 0.4982 | 0.697527 |

These values are reported in the manuscript and are not freshly computed from the files in this repository. The custom PyTorch embedding, the NB-VAE and Harmony embeddings, and the saved evaluation CSV are not included, so the comparison cannot currently be independently regenerated. The archived run_data_JQ notebook contains outputs from a different execution whose scores differ from the manuscript table; those outputs are not presented here as the manuscript run.

In this dataset each sample corresponds to one treatment, so batch and treatment labels encode the same three groups. Bio ASW is therefore confounded with batch and cannot independently establish preservation of biological signal.


### Step 5.3 Embedding Evaluation — Quantitative Comparison of All Three Methods

This evaluation compares the custom NB-VAE, PCA + Harmony, and custom PyTorch scVI-style VAE embeddings. The scVI-style method input is `5.2.3_scVI/scvi_custom/merged_scvi_custom.h5ad`; it is not the separately implemented scvi-tools SCVI model.

**Two competing goals in batch integration:**
- **Batch correction**: cells from different batches should intermix (technical variation removed)
- **Biology preservation**: cells with different biology (treatment) should still separate

**Metrics used:**

| Metric | Formula | Want | Measures |
|---|---|---|---|
| Batch ASW | `\|silhouette(X_emb, batch)\|` | **LOW** (→0) | Batch mixing |
| Bio ASW | `silhouette(X_emb, treatment)` | **HIGH** (→1) | Biological separation |
| kNN batch purity | fraction of k-NNs from same batch | **LOW** (→1/n_batches) | Local batch mixing |
| **Overall score** | `0.4×(1−batch_ASW) + 0.6×((bio_ASW+1)/2)` | **HIGH** (→1) | Batch correction plus normalized treatment-label separation |

The code normalizes Bio ASW from [−1, 1] to [0, 1] as `(Bio ASW + 1)/2` before combining it with batch correction. In this dataset each sample maps to exactly one treatment, so batch and treatment labels encode the same three groups; Bio ASW is therefore confounded with batch and cannot independently establish preservation of biological signal. The custom PyTorch scVI-style embedding is the representation used for the manuscript-reported comparison. The separate scvi-tools implementation is not used for those reported values.


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder

# ── Paths ──────────────────────────────────────────────────────────────────────
base_dir  = str(data_path('output'))
hvgs_h5ad = str(data_path('output/5.1_merged_hvgs/merged_hvgs.h5ad'))
out_dir   = os.path.join(base_dir, "5.3_eval")
os.makedirs(out_dir, exist_ok=True)

METHODS = {
    "NB-VAE\n(5.2.1)":    os.path.join(base_dir, "5.2.1_NBVAE",   "merged_nbvae.h5ad"),
    "Harmony\n(5.2.2)":   os.path.join(base_dir, "5.2.2_PCA_Harmony", "merged_harmony.h5ad"),
    "Custom PyTorch scVI-style VAE\n(5.2.3)":      os.path.join(base_dir, "5.2.3_scVI", "scvi_custom", "merged_scvi_custom.h5ad"),
}
KNN_K      = 15     # neighbors for kNN batch purity
BIO_COL    = "treatment"   # biological variable to preserve
BATCH_COL  = "batch"       # technical variable to correct
N_SUBSAMP  = 5000          # subsample for silhouette (full dataset is slow)
RANDOM_STATE = 0

# ── Load base obs (shared metadata) ───────────────────────────────────────────
print("Loading base metadata ...")
adata_base = sc.read_h5ad(hvgs_h5ad)
obs = adata_base.obs[[BATCH_COL, BIO_COL]].copy()
print(f"  {len(obs)} cells  |  batches: {obs[BATCH_COL].nunique()}  |  treatments: {obs[BIO_COL].nunique()}")

# Encode labels for silhouette
le_batch = LabelEncoder().fit(obs[BATCH_COL].astype(str))
le_bio   = LabelEncoder().fit(obs[BIO_COL].astype(str))
batch_labels = le_batch.transform(obs[BATCH_COL].astype(str))
bio_labels   = le_bio.transform(obs[BIO_COL].astype(str))

# Subsample indices for silhouette (same subset for all methods)
rng = np.random.default_rng(RANDOM_STATE)
sub_idx = rng.choice(len(obs), min(N_SUBSAMP, len(obs)), replace=False)
sub_idx_sorted = np.sort(sub_idx)

# ── Compute metrics ────────────────────────────────────────────────────────────
def knn_batch_purity(X_emb, batch_labels, k=KNN_K):
    """Fraction of k-NNs from the same batch (lower = better mixing)."""
    nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm="auto").fit(X_emb)
    _, indices = nbrs.kneighbors(X_emb)
    indices = indices[:, 1:]  # exclude self
    purity = np.mean([
        np.mean(batch_labels[idx] == batch_labels[i])
        for i, idx in enumerate(indices)
    ])
    return purity

results    = {}
raw_scores = {}   # full-precision scores for tie-free best selection
umaps      = {}

for method_name, h5ad_path in METHODS.items():
    print(f"\nEvaluating {method_name.replace(chr(10), ' ')} ...")
    if not os.path.exists(h5ad_path):
        print(f"  ✗ File not found: {h5ad_path}")
        continue

    adata = sc.read_h5ad(h5ad_path)
    if "X_emb" not in adata.obsm:
        print(f"  ✗ X_emb not found in {h5ad_path}")
        continue

    X_emb = adata.obsm["X_emb"]
    X_sub = X_emb[sub_idx_sorted]

    # Silhouette scores
    batch_asw = abs(silhouette_score(X_sub, batch_labels[sub_idx_sorted], random_state=RANDOM_STATE))
    bio_asw   = silhouette_score(X_sub,   bio_labels[sub_idx_sorted],   random_state=RANDOM_STATE)

    # kNN batch purity (subsample for speed)
    purity = knn_batch_purity(X_sub, batch_labels[sub_idx_sorted])

    # Overall score: balance batch removal and bio preservation
    batch_correction = 1.0 - batch_asw          # higher = better mixing
    bio_preservation = (bio_asw + 1.0) / 2.0    # normalize [-1,1] → [0,1]
    overall = 0.4 * batch_correction + 0.6 * bio_preservation

    results[method_name] = {
        "Batch ASW (↓)":         round(batch_asw, 4),
        "Bio ASW (↑)":           round(bio_asw, 4),
        "kNN batch purity (↓)":  round(purity, 4),
        "Batch correction (↑)":  round(batch_correction, 4),
        "Bio preservation (↑)":  round(bio_preservation, 4),
        "Overall score (↑)":     round(overall, 4),
    }
    raw_scores[method_name] = overall
    print(f"  Batch ASW = {batch_asw:.4f}  |  Bio ASW = {bio_asw:.4f}  |  kNN purity = {purity:.4f}  |  Overall = {overall:.4f}")

    # Compute UMAP for this embedding
    adata_u = sc.AnnData(X=X_emb, obs=obs)
    sc.pp.neighbors(adata_u, use_rep="X", n_neighbors=15)
    sc.tl.umap(adata_u, random_state=RANDOM_STATE)
    umaps[method_name] = adata_u.obsm["X_umap"].copy()

# ── Summary table ──────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T
print("\n=== Quantitative evaluation summary ===")
display(results_df)
results_df.to_csv(os.path.join(out_dir, "embedding_evaluation.csv"))
print(f"Saved: {out_dir}/embedding_evaluation.csv")

if raw_scores:
    best = max(raw_scores, key=raw_scores.get)
else:
    best = None
    print("\nNo evaluated embeddings were found; raw_scores is empty. Skipping best selection.")
if best is not None:
    print(f"\n\u2605  Best overall: {best.replace(chr(10), ' ')}  (score={raw_scores[best]:.6f})")
else:
    print("\nNo best embedding could be selected because no embeddings were evaluated.")
print("  All scores (full precision):", {k.replace(chr(10)," "): round(v,6) for k, v in raw_scores.items()})

# ── UMAP comparison plot ───────────────────────────────────────────────────────
n_methods = len(umaps)
fig = plt.figure(figsize=(5 * n_methods, 10))
# ── UMAP comparison plot ───────────────────────────────────────────────────────
n_methods = len(umaps)
if n_methods == 0:
    print("No evaluated embeddings were found; skipping UMAP comparison plot.")
else:
    fig = plt.figure(figsize=(5 * n_methods, 10))
    gs = gridspec.GridSpec(2, n_methods, figure=fig, hspace=0.35, wspace=0.1)

    # Color encoding
    batch_unique  = le_batch.classes_
    bio_unique    = le_bio.classes_
    pal_batch = plt.cm.tab10(np.linspace(0, 1, len(batch_unique)))
    pal_bio   = plt.cm.Set1 (np.linspace(0, 1, len(bio_unique)))
    b2c = {b: pal_batch[i] for i, b in enumerate(batch_unique)}
    t2c = {t: pal_bio[i]   for i, t in enumerate(bio_unique)}
    cell_batch_colors     = [b2c[b] for b in obs[BATCH_COL].astype(str)]
    cell_bio_colors       = [t2c[t] for t in obs[BIO_COL].astype(str)]

    for col, (method_name, coords) in enumerate(umaps.items()):
        ...
    # rest of plot code

# Color encoding
batch_unique  = le_batch.classes_
bio_unique    = le_bio.classes_
pal_batch = plt.cm.tab10(np.linspace(0, 1, len(batch_unique)))
pal_bio   = plt.cm.Set1 (np.linspace(0, 1, len(bio_unique)))
b2c = {b: pal_batch[i] for i, b in enumerate(batch_unique)}
t2c = {t: pal_bio[i]   for i, t in enumerate(bio_unique)}
cell_batch_colors     = [b2c[b] for b in obs[BATCH_COL].astype(str)]
cell_bio_colors       = [t2c[t] for t in obs[BIO_COL].astype(str)]

for col, (method_name, coords) in enumerate(umaps.items()):
    # Row 0: colored by batch
    ax0 = fig.add_subplot(gs[0, col])
    ax0.scatter(coords[:, 0], coords[:, 1], c=cell_batch_colors, s=0.5, alpha=0.4, rasterized=True)
    ax0.set_title(f"{method_name}\n(batch)", fontsize=10)
    ax0.axis("off")
    score = results[method_name]["Overall score (↑)"]
    star = " ★" if method_name == best else ""
    ax0.set_xlabel(f"Overall: {score:.3f}{star}", fontsize=9)

    # Row 1: colored by treatment (biology)
    ax1 = fig.add_subplot(gs[1, col])
    ax1.scatter(coords[:, 0], coords[:, 1], c=cell_bio_colors, s=0.5, alpha=0.4, rasterized=True)
    ax1.set_title(f"{method_name}\n(treatment)", fontsize=10)
    ax1.axis("off")

# Shared legends
batch_handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=b2c[b], markersize=7, label=b) for b in batch_unique]
bio_handles   = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=t2c[t], markersize=7, label=t) for t in bio_unique]
fig.legend(batch_handles, batch_unique, title=BATCH_COL,  loc="lower left",  ncol=len(batch_unique), fontsize=8, framealpha=0.8)
fig.legend(bio_handles,   bio_unique,   title=BIO_COL,    loc="lower right", ncol=len(bio_unique),   fontsize=8, framealpha=0.8)

fig.suptitle("Embedding comparison: batch correction (row 1) vs biology preservation (row 2)", fontsize=12)
plt.savefig(os.path.join(out_dir, "embedding_comparison_umap.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print(f"Saved: {out_dir}/embedding_comparison_umap.png")

# ── Bar chart of key metrics ───────────────────────────────────────────────────
plot_cols = ["Batch ASW (↓)", "Bio ASW (↑)", "Overall score (↑)"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ["#e74c3c" if n == best else "#7fb3d3" for n in results_df.index]
for ax, col in zip(axes, plot_cols):
    if results_df.empty:
        print("No quantitative results to plot; skipping metrics bar chart.")
        break
    vals = results_df[col]
    bars = ax.bar(range(len(vals)), vals, color=colors)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels([n.replace("\n", " ") for n in vals.index], fontsize=8)
    ax.set_title(col, fontsize=10)
    ax.set_ylim(0, max(vals.max() * 1.2, 0.1))
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    arrow = "↓ better" if "↓" in col else "↑ better"
    ax.set_ylabel(arrow, fontsize=8)

plt.suptitle("Quantitative embedding evaluation (red = best overall)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "embedding_comparison_metrics.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print(f"Saved: {out_dir}/embedding_comparison_metrics.png")
print("\nStep 5.3 Embedding Evaluation — done.")

### Step 6.1 kNN Graph — custom PyTorch scVI-style latent embedding

In [ ]:
import os
import scanpy as sc
from threadpoolctl import threadpool_limits

# ── Parameters ─────────────────────────────────────────────────────────────────
use_rep   = "X_emb"   # custom PyTorch scVI-style latent embedding
n_pcs     = 30        # latent dimensions to use
n_threads = 4

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad = str(data_path('output/5.2.3_scVI/scvi_custom/merged_scvi_custom.h5ad'))
knn_dir    = str(data_path('output/6_cluster/6.1_knn'))
os.makedirs(knn_dir, exist_ok=True)

threadpool_limits(n_threads)
sc.settings.n_jobs = n_threads

adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells x {adata.n_vars} genes")
print(f"  X_emb  : {adata.obsm['X_emb'].shape}")

sc.pp.neighbors(adata, use_rep=use_rep, n_pcs=n_pcs)
print(f"  kNN graph built  (use_rep={use_rep}, n_pcs={n_pcs})")

adata.write_h5ad(os.path.join(knn_dir, "merged_knn.h5ad"))
print(f"  Saved  : {knn_dir}/merged_knn.h5ad")
print("Step 6.1 kNN — done.")

### Step 6.2 UMAP Embedding

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc

# ── Parameters ─────────────────────────────────────────────────────────────────
n_components = 2
random_state = 0
round_digits = 10   # for hash-stable coordinates

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad = str(data_path('output/6_cluster/6.1_knn/merged_knn.h5ad'))
umap_dir   = str(data_path('output/6_cluster/6.2_umap'))
os.makedirs(umap_dir, exist_ok=True)

adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells")

sc.tl.umap(adata, n_components=n_components, random_state=random_state)
adata.obsm["X_umap"] = np.round(adata.obsm["X_umap"], round_digits)
print(f"  UMAP shape : {adata.obsm['X_umap'].shape}")

adata.write_h5ad(os.path.join(umap_dir, "merged_umap.h5ad"))
pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names).to_pickle(
    os.path.join(umap_dir, "merged_umap.pkl")
)
print(f"  Saved  : {umap_dir}/merged_umap.h5ad | merged_umap.pkl")
print("Step 6.2 UMAP — done.")

### Step 6.3 Leiden Clustering (resolution 0.5 and 1.0)

In [ ]:
import os
import matplotlib.pyplot as plt
import scanpy as sc

# ── Parameters ─────────────────────────────────────────────────────────────────
resolutions = [0.5, 1.0]

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad = str(data_path('output/6_cluster/6.2_umap/merged_umap.h5ad'))
leiden_dir = str(data_path('output/6_cluster/6.3_leiden'))
os.makedirs(leiden_dir, exist_ok=True)

adata = sc.read_h5ad(input_h5ad)
print(f"  Loaded : {adata.n_obs} cells")

for res in resolutions:
    key = f"leiden_{res}"
    %pip install leidenalg python-igraph
    sc.tl.leiden(adata, resolution=res, key_added=key)
    n_clusters = adata.obs[key].nunique()
    print(f"  Resolution {res} → {n_clusters} clusters  (key: '{key}')")

    # UMAP plot coloured by clusters
    sc.pl.umap(adata, color=key, title=f"Leiden (res={res})", show=False)
    plt.savefig(os.path.join(leiden_dir, f"merged_leiden_r{res}.png"), dpi=150, bbox_inches="tight")
    plt.close()

    # Save per-resolution cluster labels
    adata.obs[[key]].to_pickle(os.path.join(leiden_dir, f"merged_leiden_r{res}.pkl"))

# Save single h5ad with both leiden columns
adata.write_h5ad(os.path.join(leiden_dir, "merged_leiden.h5ad"))
print(f"  Saved  : {leiden_dir}/merged_leiden.h5ad")
print(f"  Cluster columns: {[f'leiden_{r}' for r in resolutions]}")
print("Step 6.3 Leiden — done.")

### Step 6.4 Entropy Scoring

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.stats import entropy

# ── Parameters ─────────────────────────────────────────────────────────────────
entropy_col  = "batch"          # column holding batch labels
resolutions  = [0.5, 1.0]      # must match Step 6.3

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad  = str(data_path('output/6_cluster/6.3_leiden/merged_leiden.h5ad'))
entropy_dir = str(data_path('output/6_cluster/6.4_entropy'))
os.makedirs(entropy_dir, exist_ok=True)

adata = sc.read_h5ad(input_h5ad)
batches = adata.obs[entropy_col].unique().tolist()
K = len(batches)
print(f"  Loaded : {adata.n_obs} cells | batches (K={K}) : {batches}")

def cluster_entropy(obs_df, cluster_col, entropy_col, K):
    """Shannon entropy (normalized by log2(K)) per cluster."""
    records = []
    for cluster, grp in obs_df.groupby(cluster_col):
        counts = grp[entropy_col].value_counts()
        probs  = counts / counts.sum()
        H      = entropy(probs, base=2)
        H_norm = H / np.log2(K) if K > 1 else 0.0
        records.append({"cluster": cluster, "entropy": H, "entropy_norm": H_norm, "n_cells": len(grp)})
    return pd.DataFrame(records).sort_values("cluster")

all_entropy = {}
for res in resolutions:
    key = f"leiden_{res}"
    df  = cluster_entropy(adata.obs, key, entropy_col, K)
    all_entropy[res] = df
    print(f"\n  Resolution {res}:")
    print(df.to_string(index=False))

    # Save per-resolution table
    df.to_pickle(os.path.join(entropy_dir, f"entropy_r{res}.pkl"))
    df.to_csv(os.path.join(entropy_dir, f"entropy_r{res}.csv"), index=False)

    # Barplot of normalized entropy per cluster
    fig, ax = plt.subplots(figsize=(max(6, len(df) * 0.5), 4))
    ax.bar(df["cluster"].astype(str), df["entropy_norm"], color="steelblue", edgecolor="white")
    ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.8, label="max entropy")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Normalized Shannon Entropy")
    ax.set_title(f"Batch Entropy per Cluster (res={res})")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(entropy_dir, f"entropy_r{res}.png"), dpi=150, bbox_inches="tight")
    plt.close()

# Attach per-cell entropy (from cluster membership) back to adata
for res in resolutions:
    key     = f"leiden_{res}"
    ent_map = all_entropy[res].set_index("cluster")["entropy_norm"]
    adata.obs[f"entropy_norm_{res}"] = adata.obs[key].map(ent_map)

adata.write_h5ad(os.path.join(entropy_dir, "merged_entropy.h5ad"))
print(f"\n  Saved  : {entropy_dir}/merged_entropy.h5ad")
print("Step 6.4 Entropy — done.")

### Step 6.5 Transfer cluster metadata to full-gene object

In [ ]:
import os
import scanpy as sc

# ── Paths ──────────────────────────────────────────────────────────────────────
full_h5ad    = str(data_path('output/5.1_merged_hvgs/merged_hvgs_full.h5ad'))
cluster_h5ad = str(data_path('output/6_cluster/6.4_entropy/merged_entropy.h5ad'))
annot_dir    = str(data_path('output/6_cluster/6.5_prep_celltypist'))
os.makedirs(annot_dir, exist_ok=True)

# ── Load ───────────────────────────────────────────────────────────────────────
full    = sc.read_h5ad(full_h5ad)
cluster = sc.read_h5ad(cluster_h5ad)
print(f"  full    : {full.n_obs} cells x {full.n_vars} genes")
print(f"  cluster : {cluster.n_obs} cells x {cluster.n_vars} genes")

# ── Transfer obs columns from cluster → full (aligned by barcode) ──────────────
transfer_obs = ["leiden_0.5", "leiden_1.0", "entropy_norm_0.5", "entropy_norm_1.0"]
for col in transfer_obs:
    full.obs[col] = cluster.obs.loc[full.obs_names, col]

# ── Transfer UMAP and custom PyTorch scVI-style latent embedding ────────────────────────────────────
full.obsm["X_umap"] = cluster.obsm["X_umap"][cluster.obs_names.get_indexer(full.obs_names)]
full.obsm["X_emb"]  = cluster.obsm["X_emb"][cluster.obs_names.get_indexer(full.obs_names)]

print(f"  Transferred obs : {transfer_obs}")
print(f"  Transferred obsm: X_umap {full.obsm['X_umap'].shape}, X_emb {full.obsm['X_emb'].shape}")
print(f"  leiden_0.5  : {full.obs['leiden_0.5'].nunique()} clusters")
print(f"  leiden_1.0  : {full.obs['leiden_1.0'].nunique()} clusters")

# ── Save ───────────────────────────────────────────────────────────────────────
out_path = os.path.join(annot_dir, "merged_pre_celltypist.h5ad")
full.write_h5ad(out_path)
print(f"\n  Saved  : {out_path}")
print(f"  Ready for cell type annotation using all {full.n_vars} genes")
print("Step 6.5 — done.")

## Step 7 — Cell Type Annotation with CellTypist

Annotate clusters using the **Mouse Whole Brain** atlas model (`Mouse_Whole_Brain.pkl`).

**Input:** `merged_full_clustered.h5ad` from Step 6.5 (full-gene matrix with Leiden clusters, NOT yet cell-type annotated).  
**Model:** `/path/to/file.pkl`  
**Output:** `merged_celltypist.h5ad` saved under `7.cell_annotation/` with `celltypist:*` columns added to `adata.obs`.

> **Note on gene names:** CellTypist requires gene symbols as `var_names`. If your `var_names` are  
> Ensembl IDs, uncomment the `adata_ct.var_names = ...` line and set `symbol_col` to the correct column.

In [ ]:
!pip install mygene

In [ ]:
import os
import pandas as pd
import scanpy as sc
import celltypist
from celltypist import models as ct_models
import mygene

# ── Paths ──────────────────────────────────────────────────────────────────────
input_h5ad  = str(data_path('output/6_cluster/6.5_prep_celltypist/merged_pre_celltypist.h5ad'))
model_path  = str(atlas_path()) #### please change to your local model path
annot_dir   = str(data_path('output/7.cell_annotation'))
out_h5ad    = os.path.join(annot_dir, "merged_celltypist_immune_low.h5ad")
out_obs_pkl = os.path.join(annot_dir, "celltypist_obs.pkl")
os.makedirs(annot_dir, exist_ok=True)

# ── Load AnnData ───────────────────────────────────────────────────────────────
adata = sc.read_h5ad(input_h5ad)
print(f"Loaded: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"var_names (first 5): {list(adata.var_names[:5])}")

# ── Prepare a copy for CellTypist (requires normalised log counts) ─────────────
adata_ct = adata.copy()
sc.pp.normalize_per_cell(adata_ct, counts_per_cell_after=1e4)
sc.pp.log1p(adata_ct)

# If var_names are Ensembl IDs, switch to gene symbols before annotating.
# Uncomment and set symbol_col to the relevant column in adata.var (e.g. "gene_symbols").
# symbol_col = "gene_symbols"
# adata_ct.var_names = adata_ct.var[symbol_col]

# ── Load local model ───────────────────────────────────────────────────────────
model_name = os.path.basename(model_path)[:-4]   # "Mouse_Whole_Brain"
model_obj  = ct_models.Model.load(model_path)
print(f"Model loaded: {model_name}")
print(f"Model cell types: {len(model_obj.cell_types)}")

# ── Run annotation ─────────────────────────────────────────────────────────────
# Make sure the annotation object uses gene symbols, not Ensembl IDs.
# CellTypist requires `var_names` to match model features.
symbol_col = None
for c in ("gene_symbols", "symbol", "gene_symbol", "Symbol", "GeneSymbol"):
    if c in adata_ct.var.columns:
        symbol_col = c
        break

if symbol_col is not None:
    adata_ct.var_names = adata_ct.var[symbol_col].astype(str)
else:
    adata_ct.var_names = adata_ct.var_names.astype(str)

# Keep only genes present in the loaded CellTypist model.
model_features = set(model_obj.features)
overlap_mask = [g in model_features for g in adata_ct.var_names]
if not any(overlap_mask):
    # CellTypist features are stored as gene symbols; many AnnData objects use
    # a different case or a symbol column. Try a case-insensitive alignment first.
    model_features_uc = {str(g).upper() for g in model_obj.features}
    var_names_uc = adata_ct.var_names.astype(str).str.upper()

    overlap_mask = [g in model_features_uc for g in var_names_uc]
    if any(overlap_mask):
        adata_ct.var_names = var_names_uc
        adata_ct = adata_ct[:, overlap_mask].copy()
    else:
        # If a dedicated symbol column exists, use it explicitly.
        if symbol_col is not None:
            adata_ct.var_names = adata_ct.var[symbol_col].astype(str).str.upper()
            overlap_mask = [g in model_features_uc for g in adata_ct.var_names]
            if any(overlap_mask):
                adata_ct = adata_ct[:, overlap_mask].copy()
            else:
                raise ValueError(
                    "No features overlap with the model after symbol remapping. "
                    "Please provide gene symbols in `adata_ct.var_names` or in a "
                    "`gene_symbols`/`symbol` column that matches the CellTypist model features."
                )
        else:
            # If the data are still using Ensembl IDs, try an Ensembl -> gene-symbol lookup.
            # This is the common case when `adata.var_names` are `ENSG...` IDs.
            if all(str(g).startswith("ENSG") for g in adata_ct.var_names[: min(100, len(adata_ct.var_names))]):
                try:
                    mg = mygene.MyGeneInfo()

                    mapping = mg.querymany(
                        list(adata_ct.var_names),
                        scopes="ensembl.gene",
                        fields="symbol",
                        species="human",
                        returnall=False,
                        verbose=False,
                    )

                    symbol_map = {}
                    for item in mapping:
                        q = item.get("query")
                        if q is None:
                            continue
                        symbol = item.get("symbol")
                        if symbol:
                            symbol_map[q] = symbol

                    if symbol_map:
                        adata_ct.var_names = pd.Index(
                            [symbol_map.get(str(g), str(g)) for g in adata_ct.var_names]
                        )
                        model_features_uc = {str(g).upper() for g in model_obj.features}
                        var_names_uc = adata_ct.var_names.astype(str).str.upper()

                        overlap_mask = [g in model_features_uc for g in var_names_uc]
                        if any(overlap_mask):
                            adata_ct = adata_ct[:, overlap_mask].copy()
                        else:
                            raise ValueError(
                                "No features overlap with the model after Ensembl-to-symbol mapping. "
                                "Please provide gene symbols in `adata_ct.var_names` or in a "
                                "`gene_symbols`/`symbol` column that matches the CellTypist model features."
                            )
                    else:
                        raise ValueError(
                            "No features overlap with the model. "
                            "Your `adata_ct.var_names` appear to be Ensembl IDs, but no gene-symbol "
                            "mapping was found. Please provide gene symbols in `adata_ct.var_names` "
                            "or in a `gene_symbols`/`symbol` column."
                        )
                except Exception:
                    raise ValueError(
                        "No features overlap with the model. "
                        "Your `adata_ct.var_names` do not match the CellTypist gene symbols "
                        "(case-insensitive overlap was also unsuccessful, and Ensembl-to-symbol "
                        "mapping was not available). Please provide gene symbols in "
                        "`adata_ct.var_names` or in a `gene_symbols`/`symbol` column."
                    ) from None
            else:
                raise ValueError(
                    "No features overlap with the model. "
                    "Your `adata_ct.var_names` do not match the CellTypist gene symbols "
                    "(case-insensitive overlap was also unsuccessful). "
                    "Please provide gene symbols in `adata_ct.var_names` or in a "
                    "`gene_symbols`/`symbol` column."
                )

# Rebuild the overlap mask against the current adata_ct shape (5782 vars),
# instead of reusing the stale mask that was computed before symbol remapping.
current_var_names = adata_ct.var_names.astype(str)
overlap_mask = [g in model_features for g in current_var_names]

if not any(overlap_mask):
    model_features_uc = {str(g).upper() for g in model_obj.features}
    overlap_mask = [str(g).upper() in model_features_uc for g in current_var_names]

if not any(overlap_mask):
    raise ValueError(
        "No features overlap with the model after current name normalization."
    )

adata_ct = adata_ct[:, overlap_mask].copy()

# Run annotation on the overlap-only object.
predictions = celltypist.annotate(
    adata_ct,
    model=model_obj,
    majority_voting=True,
)
predictions_adata = predictions.to_adata()

df_ct = predictions_adata.obs.loc[
    adata.obs.index, ["predicted_labels", "conf_score"]
].copy()
df_ct.columns = [f"celltypist:{model_name}", f"celltypist:{model_name}:conf"]

# ── Merge results back into original adata ─────────────────────────────────────
adata.obs = pd.concat([adata.obs, df_ct], axis=1)

# ── Save ───────────────────────────────────────────────────────────────────────
df_ct.to_pickle(out_obs_pkl)
# Prevent duplicate CellTypist columns if the notebook is rerun on the same AnnData.
drop_cols = [
    col for col in [f"celltypist:{model_name}", f"celltypist:{model_name}:conf"]
    if col in adata.obs.columns
]
if drop_cols:
    adata.obs = adata.obs.drop(columns=drop_cols)

adata.obs = pd.concat([adata.obs, df_ct], axis=1)

# Save the updated object (avoids repeated column-name errors on reruns)
adata.write_h5ad(out_h5ad)

print(f"\nAnnotation complete.")
print(f"  Unique cell types assigned : {adata.obs[f'celltypist:{model_name}'].nunique()}")
print(f"  Output h5ad  : {out_h5ad}")
print(f"  Output obs   : {out_obs_pkl}")

### Visualise CellTypist Annotations on UMAP

In [ ]:
import matplotlib.pyplot as plt

ct_col  = f"celltypist:{model_name}"
conf_col = f"celltypist:{model_name}:conf"

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

sc.pl.umap(adata, color=ct_col,   legend_loc="on data", legend_fontsize=6,
           title="CellTypist — predicted labels", ax=axes[0], show=False)
sc.pl.umap(adata, color=conf_col, color_map="viridis",
           title="CellTypist — confidence score",  ax=axes[1], show=False)

plt.tight_layout()
plt.savefig(os.path.join(annot_dir, "umap_celltypist_immune_low.png"), dpi=150, bbox_inches="tight")
plt.show()

# Per-cluster majority vote summary
summary = (
    adata.obs.groupby("leiden_0.5")[ct_col]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={ct_col: "majority_cell_type"})
)
print(summary.to_string(index=False))

## Split large h5ad files for GitHub upload (< 100 MB per part)

In [ ]:
import anndata as ad
import numpy as np
import os
import h5py

def read_h5ad_patched(path):
    """Workaround for anndata null-encoding bug: removes uns/log1p/base (None) before reading."""
    with h5py.File(path, 'a') as f:
        lp = f.get('uns/log1p')
        if lp is not None and 'base' in lp:
            del lp['base']
    return ad.read_h5ad(path)

# --- Split merged_integrate.h5ad ---
input_path = "../test_data/output/merged_integrate.h5ad"
output_dir = "../test_data/output/merged_integrate_parts"
os.makedirs(output_dir, exist_ok=True)

adata = read_h5ad_patched(input_path)
print(f"Loaded merged_integrate: {adata.n_obs} cells x {adata.n_vars} genes")

n_parts = 11  # ~520 MB / 11 ≈ 47 MB each (under 50 MB GitHub limit)
indices = np.array_split(np.arange(adata.n_obs), n_parts)

for i, idx in enumerate(indices):
    chunk = adata[idx].copy()
    out_path = os.path.join(output_dir, f"merged_integrate_part{i+1}.h5ad")
    chunk.write_h5ad(out_path)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f"  Part {i+1}: {len(idx)} cells → {out_path} ({size_mb:.1f} MB)")

print("Done splitting merged_integrate.h5ad")

In [ ]:
# --- Split merged_entropy_full.h5ad ---
input_path = "../test_data/output/merged_entropy_full.h5ad"
output_dir = "../test_data/output/merged_entropy_full_parts"
os.makedirs(output_dir, exist_ok=True)

adata = read_h5ad_patched(input_path)
print(f"Loaded merged_entropy_full: {adata.n_obs} cells x {adata.n_vars} genes")

n_parts = 11  # ~524 MB / 11 ≈ 47 MB each (under 50 MB GitHub limit)
indices = np.array_split(np.arange(adata.n_obs), n_parts)

for i, idx in enumerate(indices):
    chunk = adata[idx].copy()
    out_path = os.path.join(output_dir, f"merged_entropy_full_part{i+1}.h5ad")
    chunk.write_h5ad(out_path)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f"  Part {i+1}: {len(idx)} cells → {out_path} ({size_mb:.1f} MB)")

print("Done splitting merged_entropy_full.h5ad")

## Combine h5ad parts back into single files

In [ ]:
import anndata as ad
import os

n_parts = 11

# --- Combine merged_integrate parts ---
parts_dir = str(data_path('intermediate_output'))

parts = [ad.read_h5ad(os.path.join(parts_dir, f"merged_integrate_part{i+1}.h5ad")) for i in range(n_parts)]
adata_combined = ad.concat(parts, merge="same")
adata_combined.write_h5ad(str(data_path('output/merged_integrate_restored.h5ad')))
print(f"Restored merged_integrate: {adata_combined.n_obs} cells x {adata_combined.n_vars} genes")

'''
# --- Combine merged_entropy_full parts ---
parts_dir = "../test_data/output/merged_entropy_full_parts"

parts = [ad.read_h5ad(os.path.join(parts_dir, f"merged_entropy_full_part{i+1}.h5ad")) for i in range(n_parts)]
adata_combined = ad.concat(parts, merge="same")
adata_combined.write_h5ad("../test_data/output/merged_entropy_full_restored.h5ad")
print(f"Restored merged_entropy_full: {adata_combined.n_obs} cells x {adata_combined.n_vars} genes")
'''